# 航空公司 AI 客服（工具调用 + SQLite + Gradio）

## 练习目标

做一个「会查票价、也会学新票价」的航空助手：

- **存储**：用本地 **SQLite** 表 `flights` 保存城市 → 价格
- **推理**：LLM 通过 **Function Calling / Tool Use** 调用 `get_price` / `add_price`
- **界面**：用 **Gradio** `ChatInterface` 做聊天 UI
- **网关**：OpenAI 兼容客户端指向 **OpenRouter**（`base_url` 固定）

## 和本课 Week 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Tool schema | `tools = [{type: function, ...}]` |
| Agent loop | `while True`：有 `tool_calls` 就执行再问模型 |
| 外部状态 | SQLite 读写，让助手「记住」新价格 |
| Chat UI | Gradio 把用户消息交给代理循环 |

## 怎么跑

1. `.env` 里准备 `OPENROUTER_API_KEY`
2. 从上到下运行单元格（会创建 `flights.db` 并写入种子数据）
3. 最后一格启动 Gradio；可问「London 多少钱？」或教它新城市价格


In [ ]:
# ========== 导入与客户端：OpenRouter + Gradio + SQLite ==========

# 标准库 os：读环境变量（Environment Variables），例如 OpenRouter API Key
import os
# 标准库 json：把工具返回结果序列化成字符串，塞进 role=tool 的 content
import json
# 标准库 sqlite3：轻量本地数据库，存城市机票价格
import sqlite3
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类（这里走 OpenRouter 的兼容接口）
from openai import OpenAI
# 导入 gradio：快速搭聊天 Web UI（ChatInterface）
import gradio as gr

# 加载 .env（无额外参数，保持原行为）
load_dotenv()

# 从环境变量读取 OpenRouter 密钥；变量名必须是 OPENROUTER_API_KEY
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# 创建 OpenAI 兼容客户端：api_key + base_url 指向 OpenRouter
client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

# 模型 id（OpenRouter 路由名）：openai/gpt-4o-mini —— 字符串勿改
MODEL = "openai/gpt-4o-mini"


## 数据库设置（SQLite）

创建本地文件 `flights.db`，表结构很简单：

- `city`：城市名（主键）
- `price`：整数票价

`CREATE TABLE IF NOT EXISTS`：表已存在时不会重建，方便反复跑笔记本。


In [ ]:
# ========== 建库建表：本地 flights.db ==========

# 连接 SQLite；check_same_thread=False 允许 Gradio 多线程复用同一连接（原参数保留）
conn = sqlite3.connect("flights.db", check_same_thread=False)
# 游标：后续 execute / fetch 都走它
cursor = conn.cursor()

# 创建 flights 表（若不存在）：city 主键 + price 整数
cursor.execute("""
CREATE TABLE IF NOT EXISTS flights (
    city TEXT PRIMARY KEY,
    price INTEGER
)
""")

# 提交 DDL，让表结构落盘
conn.commit()


## 工具函数（给 LLM 真正执行的 Python）

这两个函数是 **Tool Use** 的后端实现：

- `get_price(cities)`：批量查价；没有记录时返回字面量 `"NOT_FOUND"`（system prompt 会据此追问用户）
- `add_price(city, price)`：写入或覆盖价格（`INSERT OR REPLACE`），实现「用户教助手」


In [ ]:
# ========== 工具实现：查价 / 写价 ==========

# 按城市列表查票价；返回 dict：城市 → 价格或 "NOT_FOUND"
def get_price(cities):
    # 结果字典，边查边填
    results = {}
    # 逐个城市查询（参数化 SQL，避免拼接注入）
    for city in cities:
        # ? 占位符绑定 city
        cursor.execute("SELECT price FROM flights WHERE city = ?", (city,))
        # fetchone：有行则是 (price,)；没有则是 None
        result = cursor.fetchone()

        if result:
            # 取元组第一个元素（price）
            results[city] = result[0]
        else:
            # 关键哨兵字符串：供模型/提示词判断「库里没有」——勿改字面量
            results[city] = "NOT_FOUND"   # 🔥 KEY FIX

    return results


# 新增或覆盖某城市票价；price 必须为正
def add_price(city, price):
    # 非法价格直接返回错误文案（可运行字符串保留英文）
    if price <= 0:
        return "Invalid price"

    # INSERT OR REPLACE：主键冲突则覆盖旧价
    cursor.execute(
        "INSERT OR REPLACE INTO flights (city, price) VALUES (?, ?)",
        (city, price)
    )
    # 写操作要 commit，否则别的连接看不到
    conn.commit()
    # 返回给人看的确认字符串（保留原英文格式）
    return f"Saved {city} → ${price}"


## 种子数据（Seed）

预置几个常见城市票价，方便一打开就能演示「查价」。

`INSERT OR IGNORE`：主键已存在则跳过，反复运行不会重复插入或覆盖你后来教的价格。


In [ ]:
# ========== 种子数据：演示用默认城市价格 ==========

# 写入默认城市列表；已存在则忽略（IGNORE）
def seed_data():
    # (城市名, 价格) 元组列表；城市名字符串保持英文，与后续提问一致
    default_data = [
        ("London", 500),
        ("Paris", 450),
        ("New York", 700),
        ("Tokyo", 800),
        ("Dubai", 300)
    ]

    # 逐条插入；OR IGNORE 避免重复
    for city, price in default_data:
        cursor.execute(
            "INSERT OR IGNORE INTO flights (city, price) VALUES (?, ?)",
            (city, price)
        )

    # 提交事务
    conn.commit()
    # 提示种子已处理（可能本就存在）
    print("✅ Seed data inserted (if not already present)")


# 定义完立刻执行一次，保证库里有演示数据
seed_data()


## 工具定义（Tool Schema）

把上面的 Python 函数，用 **JSON Schema** 描述成模型能理解的 `tools` 列表。

要点：

- `name` 必须和真实函数名一致（`get_price` / `add_price`）
- `description` / 参数说明给模型看，影响它何时调用——**保持英文原文**
- 运行时把这个列表传给 `chat.completions.create(..., tools=tools)`


In [ ]:
# ========== tools：OpenAI / OpenRouter 函数调用 schema ==========

# 工具列表：每个元素 type=function，内含 name / description / parameters
tools = [
    {
        # 声明这是函数类工具
        "type": "function",
        "function": {
            # 工具名：必须与下面分发时的 if name == "get_price" 一致
            "name": "get_price",
            # 给模型看的说明（影响调用决策，勿翻译）
            "description": "Get flight prices for cities",
            "parameters": {
                "type": "object",
                "properties": {
                    # cities：字符串数组，可一次查多城
                    "cities": {
                        "type": "array",
                        "items": {"type": "string"}
                    }
                },
                # 必填字段
                "required": ["cities"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            # 第二个工具：写价
            "name": "add_price",
            "description": "Add or update flight price",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "price": {"type": "integer"}
                },
                "required": ["city", "price"]
            }
        }
    }
]


## 系统提示（System Prompt）

用 `SYSTEM_PROMPT` 规定助手行为：何时查价、何时追问、何时写入、用 Markdown 回复。

> 发给模型的英文规则正文**不要翻译**——改字会改变工具调用策略与口吻。


In [ ]:
# ========== SYSTEM_PROMPT：规定助手何时调用哪个工具 ==========

# 三引号字符串整段发给模型；Rules / NOT_FOUND 等字面量与工具返回值对齐，勿改
SYSTEM_PROMPT = """
You are a smart airline assistant.

Rules:
1. If user mentions a single city → assume they want price for that city.
2. Always call get_price with that city.
3. If NOT_FOUND → ask user for price.
4. If user provides price → call add_price.
5. Always respond in markdown.
"""


## 代理循环（Agent Loop）

`chat_with_agent` 是核心：

1. 拼 messages：system + 历史 + 本轮 user
2. 调模型，带上 `tools`
3. 若有 `tool_calls`：把 assistant 消息入列 → 本地执行函数 → 以 `role=tool` 回传 → `continue` 再问
4. 若无工具调用：把最终回复入列并返回文本

这就是 Week 2 的「多步工具推理」最小完整版。


In [ ]:
# ========== 代理循环：模型 ↔ 工具 直到给出最终自然语言 ==========

# user_input：本轮用户话；history：先前 messages（默认空列表）
def chat_with_agent(user_input, history=[]):
    # 每轮对话都以 system 开头，固定行为规则
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # 并入历史（若 Gradio/调用方传入了过去的 messages）
    messages.extend(history)
    # 追加本轮用户消息
    messages.append({"role": "user", "content": user_input})

    # 循环：可能多轮 tool_calls，直到模型不再要工具
    while True:
        # 调用 Chat Completions；带上 MODEL 与 tools schema
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        # 取第一条 choice 的 message（可能含 tool_calls）
        msg = response.choices[0].message

        # 仅当模型请求工具时进入分支
        if msg.tool_calls:

            # 必须先把带 tool_calls 的 assistant 消息写入历史，API 才认后续 tool 结果
            messages.append(msg)  # 🔥 IMPORTANT (assistant message with tool_calls)

            # 可能一次请求多个工具，逐个执行
            for tool_call in msg.tool_calls:
                # 工具函数名（字符串）
                name = tool_call.function.name
                # arguments 是 JSON 字符串 → dict
                args = json.loads(tool_call.function.arguments)

                if name == "get_price":
                    # **args 解包为关键字参数（cities=...）
                    result = get_price(**args)

                elif name == "add_price":
                    result = add_price(**args)

                # tool 消息必须带 tool_call_id，与上面的调用一一对应
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    # 把 Python 对象打成 JSON 字符串给模型读
                    "content": json.dumps(result)
                })

            # 带着工具结果再进下一轮 create
            continue  # loop again

        else:
            # 没有工具调用：这是最终自然语言回答
            messages.append(msg)
            # 返回展示用文本 + 完整 messages（调用方可当 history）
            return msg.content, messages


## Gradio 聊天界面

用 `gr.ChatInterface` 包一层：用户每发一条，就调用 `respond` → `chat_with_agent`。

标题与描述字符串保持英文原样（UI 可运行文案不改）。浏览器打开后即可对话测工具调用。


In [ ]:
# ========== Gradio：把代理循环接到 ChatInterface ==========

# 预留的历史列表变量（本格 respond 实际把 Gradio 的 history 传给代理）
chat_history = []

# Gradio 回调：message 是用户输入，history 是界面会话历史
def respond(message, history):
    try:
        # 跑代理循环；只要展示用的 response 文本
        response, updated_messages = chat_with_agent(message, history)
        return response
    except Exception as e:
        # 出错时返回提示字符串（文案保留原样）
        return f"⚠️ Error occurred: {str(e)}"

# 创建聊天界面：绑定 fn=respond
demo = gr.ChatInterface(
    fn=respond,
    title="✈️ Airline AI Assistant",
    description="Ask for flight prices or teach new ones!"
)

# 启动本地 Gradio 服务（默认会打印访问 URL）
demo.launch()


In [ ]:
# ========== 调试：打印当前 flights 表全部行 ==========

# 查出所有城市与价格（验证种子 / add_price 是否生效）
cursor.execute("SELECT * FROM flights")
# fetchall 得到列表 of 元组，直接打印
print(cursor.fetchall())
